In [1]:
from fragnnet.runner import load_config
from fragnnet.pl_model import FragGNNPL, NeimsPL
from fragnnet.massformer.pl_model import MassFormerPL
from fragnnet.iceberg.pl_model import IcebergIntenPL, IcebergGenPL
from fragnnet.graff.pl_model import GrAFFPL
from fragnnet.utils.misc_utils import NestedDefaultDict, booltype

import argparse
from pprint import pprint
import pytorch_lightning as pl
from IPython.display import display
import os

In [2]:
model_type_to_model_cls = {
    "neims": NeimsPL,
    "massformer": MassFormerPL,
    "fragnnet_d3": FragGNNPL,
    "fragnnet_d4": FragGNNPL,
    "iceberg_gen": IcebergGenPL,
    "iceberg_inten": IcebergIntenPL,
    "iceberg_inten_opt": IcebergIntenPL,
    "graff": GrAFFPL
}
model_types = list(model_type_to_model_cls.keys())

model_to_config = NestedDefaultDict()

# setup conigs
template_fp = "../config/template.yml"
for model_type in model_types:
    model_to_config[model_type] = f"../config/nist20_inchikey/{model_type}/s0.yml"
pprint(model_to_config)

{'neims': '../config/nist20_inchikey/neims/s0.yml', 'massformer': '../config/nist20_inchikey/massformer/s0.yml', 'fragnnet_d3': '../config/nist20_inchikey/fragnnet_d3/s0.yml', 'fragnnet_d4': '../config/nist20_inchikey/fragnnet_d4/s0.yml', 'iceberg_gen': '../config/nist20_inchikey/iceberg_gen/s0.yml', 'iceberg_inten': '../config/nist20_inchikey/iceberg_inten/s0.yml', 'iceberg_inten_opt': '../config/nist20_inchikey/iceberg_inten_opt/s0.yml', 'graff': '../config/nist20_inchikey/graff/s0.yml'}


In [3]:
model_to_param_counts = NestedDefaultDict()
model_to_summary = {}

for model_type in model_types:
    print(f">>> Starting model_type {model_type}")
    custom_fp = model_to_config[model_type]
    print(f">>> Loading config {custom_fp}")
    config_d = load_config(template_fp, custom_fp)
    config_d["accelerator"] = "cpu"
    config_d["compile"] = False
    print(f">>> Initializing model")
    model_cls = model_type_to_model_cls[model_type]
    model = model_cls(**config_d)
    if model_type in ["neims","massformer"]:
        model_summary = pl.utilities.model_summary.ModelSummary(model, max_depth=4)
        model_to_summary[model_type] = model_summary
        model_idx = model_summary.layer_names.index("model")
        output_idxs = [model_summary.layer_names.index(f"model.ffn.{k}") for k in ["forw_out_layer","rev_out_layer","out_gate"]]
        model_to_param_counts[model_type] = model_summary.param_nums[model_idx]
        model_to_param_counts[f"{model_type}_output"] = \
            model_summary.param_nums[model_idx] - sum([model_summary.param_nums[i] for i in output_idxs])
    else:
        model_summary = pl.utilities.model_summary.ModelSummary(model, max_depth=4)
        model_to_summary[model_type] = model_summary
        model_idx = model_summary.layer_names.index("model")
        model_to_param_counts[model_type] = model_summary.param_nums[model_idx]
    del model

if "iceberg" in model_to_param_counts:
    assert "iceberg_gen" in model_to_param_counts
    model_to_param_counts["iceberg_inten3_100_long"] += model_to_param_counts.pop("iceberg_gen")

if "neims" in model_to_param_counts:
    model_to_param_counts["neims_nooutput"] = model_to_param_counts["neims"] - model_to_param_counts["neims_output"]
    del model_to_param_counts["neims_output"]

if "massformer" in model_to_param_counts:
    model_to_param_counts["massformer_nooutput"] = model_to_param_counts["massformer"] - model_to_param_counts["massformer_output"]
    del model_to_param_counts["massformer_output"]


>>> Starting model_type neims
>>> Loading config ../config/nist20_inchikey/neims/s0.yml
>>> Initializing model
>>> Starting model_type massformer
>>> Loading config ../config/nist20_inchikey/massformer/s0.yml
>>> Initializing model
>>> Starting model_type fragnnet_d3
>>> Loading config ../config/nist20_inchikey/fragnnet_d3/s0.yml
>>> Initializing model
>>> Starting model_type fragnnet_d4
>>> Loading config ../config/nist20_inchikey/fragnnet_d4/s0.yml
>>> Initializing model
>>> Starting model_type iceberg_gen
>>> Loading config ../config/nist20_inchikey/iceberg_gen/s0.yml
>>> Initializing model
>>> Starting model_type iceberg_inten
>>> Loading config ../config/nist20_inchikey/iceberg_inten/s0.yml
>>> Initializing model
>>> Starting model_type iceberg_inten_opt
>>> Loading config ../config/nist20_inchikey/iceberg_inten_opt/s0.yml
>>> Initializing model
>>> Starting model_type graff
>>> Loading config ../config/nist20_inchikey/graff/s0.yml
>>> Initializing model


In [4]:
model_to_summary["massformer"]

   | Name                                      | Type                           | Params
----------------------------------------------------------------------------------------------
0  | model                                     | MassFormerModel                | 165 M 
1  | model.gf                                  | GFv2Embedder                   | 48.3 M
2  | model.gf.encoder                          | GraphormerEncoder              | 48.3 M
3  | model.gf.encoder.graph_encoder            | GraphormerGraphEncoder         | 47.1 M
4  | model.gf.encoder.masked_lm_pooler         | Linear                         | 590 K 
5  | model.gf.encoder.lm_head_transform_weight | Linear                         | 590 K 
6  | model.gf.encoder.layer_norm               | LRPLayerNorm                   | 1.5 K 
7  | model.ce_embedder                         | Sequential                     | 1.9 K 
8  | model.ce_embedder.0                       | FourierFeaturizerAbsoluteSines | 1.6 K 
9  | model.ce_e

In [5]:
model_to_param_counts_m = {k: f"{v/1e6:.1f}" for k,v in model_to_param_counts.items()}


pprint(model_to_param_counts_m)

{'fragnnet_d3': '1.2',
 'fragnnet_d4': '1.6',
 'graff': '57.8',
 'iceberg_gen': '3.6',
 'iceberg_inten': '14.0',
 'iceberg_inten_opt': '13.5',
 'massformer': '165.3',
 'massformer_nooutput': '116.0',
 'neims': '125.4',
 'neims_nooutput': '116.4'}
